In [ ]:
# IMPORTANT: Due to Windows dynamic library / runtime state collision torch must be imported before mlrun
import torch
import mlrun

# Loads vars including: AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
from dotenv import load_dotenv
load_dotenv() 


from pathlib import Path
artifact_path = Path.cwd().parent #/ "mlrun-data/"
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path

p = mlrun.set_environment(api_path="http://localhost:8080", artifact_path=artifact_path)

project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory


# Run the workflow

In [ ]:
# passing this in the argument hyperparams, will loop over every combination
# hyperparameters = {'epochs': [10, 20],
#                    'lr': [10, 20]}

run_obj = project.run_function(
    function="eval-Train",    # use the name in the register.ipynb file
    params={
        "train_dataset":"raw-proc-process-raw_test_data",
        "train_dataset_tag":"20260506_1224",

        "val_dataset":"raw-proc-process-raw_validation_data",
        "val_dataset_tag":"20260506_1224",

        "test_dataset":"raw-proc-process-raw_test_data",
        "test_dataset_tag":"20260506_1224",

        "prompt":"contract_extractor_prompt",
        "prompt_tag":"20260508_1243",
        ## 
        "epochs":1,
        'batch_grad_accumulation':16,
        'learning_rate':2e-4, # QLoRA requires slightly higher LR
        'lora_r':16,
        'lora_alpha':32,
        'early_stopping_threshold':1e-3
    },
    local=True, # Run the pipeline sequence locally
    watch=True,  # Print the progress to the console
    verbose=True
)

In [ ]:
run_obj.outputs